# LimingjieNN · Colab 训练流水线

| 运行时 | 单元 |
| --- | --- |
| CPU Runtime | Cell 1 初始化 → Cell 2 单测及数据审计 → Cell 3 本地缓存、模拟器与对账 → Cell 4 CPU 成本短测 → Cell 9 收尾。S0 放行后用 Cell 5 分段采样，S5 用 Cell 8 回放与封存评估。 |
| T4 GPU Runtime | 重新运行 Cell 1。S0 运行 Cell 6 的 s0_timing 模式。S2 或 S4 运行 Cell 6 的 train 模式，并用 Cell 7 验收导出。每次会话结束运行 Cell 9。 |

执行顺序为 S0 → S1 → S2 → S4 → S5。启用 S3 主动挖掘时，先按 TASK.md 准备采样配置和前缀，再用 Cell 5 执行分段接力。开新运行时前先确认分片、manifest 和回执已同步，再释放原运行时。

Cell 1 默认从 Drive 的 source.zip 装载代码，也可用 Drive 的 repo/ 或私有 GitHub（Colab Secret LIMINGJIENN_GITHUB_TOKEN）。在 LOCKED_COMMIT 填写实际提交。Git 归档 ZIP 自带提交号，普通 ZIP 或无 .git 的 Drive 副本需人工核对来源。用 ROUND 选择配置，为采样填写 SAMPLE_PREFIX、ROUND_ID、SEED、TARGET_BATTLES，为训练列全 TRAIN_PREFIXES 并选择输出目录 ABLATION_DIR。

Drive 根目录为 /content/drive/MyDrive/LimingjieNN，其中 master_cn.db 为锁库，cache/ 放服务端 ZIP、wheelhouse/ 和未压缩 persistent tar，data/ 放特征目录、probe 和全部分片，runs/ 放训练结果，export_bundle/ 放 ONNX，status/ 放 S0 报告和唯一回执 latest_receipt.json。模拟器缓存和采样工作副本位于本地 /tmp。

In [ ]:
# Cell 1 · CPU / T4 · Initialize the repository and offline wheelhouse
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import tomllib
import zipfile

from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/LimingjieNN')
REPO_SOURCE = 'source_zip'  # 'source_zip', 'drive', or 'github'
SOURCE_ZIP_PATH = DRIVE_ROOT / 'source.zip'
LOCKED_COMMIT = ''  # Set the full Round 1 SHA after freezing the code.
CN_DB_PATH = DRIVE_ROOT / 'master_cn.db'
CN_DB_SHA256 = ''  # Optional pinned DB file digest.
CATALOG_SHA256 = ''  # Optional pinned feature catalog digest for verify-data.
CN_DB_VERSION = '202611131613'

if LOCKED_COMMIT and not re.fullmatch(r'[0-9a-f]{40}', LOCKED_COMMIT):
    raise ValueError('LOCKED_COMMIT must be a full lowercase Git SHA')
ACTUAL_COMMIT = ''

if REPO_SOURCE == 'source_zip':
    REPO_DIR = Path('/content/LimingjieNN-zip')
    with tempfile.TemporaryDirectory(dir='/content', prefix='limingjienn-source-') as tmp:
        staging = Path(tmp)
        with zipfile.ZipFile(SOURCE_ZIP_PATH) as archive:
            archive.extractall(staging)
            if re.fullmatch(rb'[0-9a-f]{40}', archive.comment):
                ACTUAL_COMMIT = archive.comment.decode('ascii')
        if (staging / 'pyproject.toml').is_file():
            source = staging
        else:
            candidates = [p for p in staging.iterdir() if p.is_dir() and (p / 'pyproject.toml').is_file()]
            if len(candidates) != 1:
                raise ValueError(f'Could not locate a single repository in {SOURCE_ZIP_PATH}')
            source = candidates[0]
        if LOCKED_COMMIT and ACTUAL_COMMIT and LOCKED_COMMIT != ACTUAL_COMMIT:
            raise ValueError(f'ZIP commit {ACTUAL_COMMIT} differs from {LOCKED_COMMIT}')
        if REPO_DIR.is_dir():
            shutil.rmtree(REPO_DIR)
        source.rename(REPO_DIR)
elif REPO_SOURCE == 'drive':
    source = DRIVE_ROOT / 'repo'
    REPO_DIR = Path('/content/LimingjieNN-drive')
    if not (source / 'pyproject.toml').is_file():
        raise FileNotFoundError(f'Upload the locked repository to {source}')
    if (source / '.git').exists():
        ACTUAL_COMMIT = subprocess.check_output(['git', '-C', source, 'rev-parse', 'HEAD'], text=True).strip()
        if LOCKED_COMMIT and ACTUAL_COMMIT != LOCKED_COMMIT:
            raise ValueError(f'Drive repo commit {ACTUAL_COMMIT} differs from {LOCKED_COMMIT}')
    if REPO_DIR.is_dir():
        shutil.rmtree(REPO_DIR)
    shutil.copytree(source, REPO_DIR, ignore=shutil.ignore_patterns(
        '.git', '.venv', 'artifacts', '__pycache__', '*.egg-info',
    ))
elif REPO_SOURCE == 'github':
    from google.colab import userdata

    REPO_DIR = Path('/content/LimingjieNN')
    with tempfile.TemporaryDirectory() as auth_dir:
        askpass = Path(auth_dir) / 'git-askpass'
        askpass.write_text(
            '#!/bin/sh\ncase "$1" in\n'
            '  *Username*) printf "%s\\n" x-access-token ;;\n'
            '  *) printf "%s\\n" "$LIMINGJIENN_GITHUB_TOKEN" ;;\n'
            'esac\n', encoding='utf-8',
        )
        askpass.chmod(0o700)
        git_env = dict(os.environ, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0',
                       LIMINGJIENN_GITHUB_TOKEN=userdata.get('LIMINGJIENN_GITHUB_TOKEN'))
        git = ['git', '-C', REPO_DIR]
        if not (REPO_DIR / '.git').is_dir():
            subprocess.run(['git', 'clone', '--branch', 'main',
                            'https://github.com/Flysky701/LimingjieNN.git', REPO_DIR], env=git_env, check=True)
        if LOCKED_COMMIT:
            subprocess.run([*git, 'fetch', 'origin', LOCKED_COMMIT], env=git_env, check=True)
            subprocess.run([*git, 'switch', '--detach', LOCKED_COMMIT], check=True)
        else:
            subprocess.run([*git, 'switch', 'main'], check=True)
            subprocess.run([*git, 'pull', '--ff-only'], env=git_env, check=True)
    del git_env
    ACTUAL_COMMIT = subprocess.check_output([*git, 'rev-parse', 'HEAD'], text=True).strip()
else:
    raise ValueError('REPO_SOURCE must be source_zip, drive, or github')

CACHE_DIR = DRIVE_ROOT / 'cache'
DATA_DIR = DRIVE_ROOT / 'data'
RUNS_DIR = DRIVE_ROOT / 'runs'
EXPORT_DIR = DRIVE_ROOT / 'export_bundle'
STATUS_DIR = DRIVE_ROOT / 'status'
for folder in (CACHE_DIR, DATA_DIR, RUNS_DIR, EXPORT_DIR, STATUS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
SOURCE_DIR = str(REPO_DIR / 'src')
os.environ.update({
    name: str(value) for name, value in {
        'DRIVE_ROOT': DRIVE_ROOT, 'CACHE_DIR': CACHE_DIR, 'DATA_DIR': DATA_DIR,
        'RUNS_DIR': RUNS_DIR, 'EXPORT_DIR': EXPORT_DIR, 'STATUS_DIR': STATUS_DIR,
        'CN_DB_PATH': CN_DB_PATH, 'CN_DB_VERSION': CN_DB_VERSION, 'PYTHONPATH': SOURCE_DIR,
    }.items()
})
subprocess.run(['bash', 'scripts/colab_install_deps.sh', '--repo-dir', REPO_DIR,
                '--wheelhouse', CACHE_DIR / 'wheelhouse'], cwd=REPO_DIR, check=True)
if SOURCE_DIR not in sys.path:
    sys.path.insert(0, SOURCE_DIR)

ROUND = 1  # 0 for Round 0 reproduction. For S3/S4, set the sampling and training inputs below.
PORT = 8080
SIM_URL = f'http://127.0.0.1:{PORT}'
CATALOG_PATH = DATA_DIR / 'feature_pack.json'
S0_PROFILE = 'configs/sim_profile_round1_cn.json'
S0_SAMPLING = 'configs/sampling_round1.json'
PROFILE = 'configs/sim_profile_highscore_cn.json' if ROUND == 0 else S0_PROFILE
SAMPLING = 'configs/sampling_round0.json' if ROUND == 0 else S0_SAMPLING
SAMPLE_PREFIX = 'round0_pilot' if ROUND == 0 else 'round1_scale'
ROUND_ID = SAMPLE_PREFIX
SEED = 20261122 if ROUND == 0 else 20261124
CHUNK_SIZE = 5000 if ROUND == 0 else 2500
TARGET_BATTLES = 25000 if ROUND == 0 else 125000
TRAIN_PREFIXES = ['round0_pilot'] if ROUND == 0 else ['round0_pilot', SAMPLE_PREFIX]
ABLATION_DIR = RUNS_DIR / f'ablation_round{ROUND}'  # Use a distinct directory for S4 inputs.
PROBE_PATH = DATA_DIR / f'probe_round{ROUND}.json'

def run_module(module, *args, check=True, **options):
    return subprocess.run([sys.executable, '-m', module, *map(str, args)],
                          cwd=REPO_DIR, check=check, **options)

def file_sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def collect_shards(prefix, directory=DATA_DIR):
    pattern = re.compile(rf'{re.escape(prefix)}(?:\.part-\d{{5,}})?\.jsonl')
    return sorted(p for p in directory.glob(f'{prefix}*.jsonl') if p.is_file() and pattern.fullmatch(p.name))

def shard_hashes(manifest):
    entries = json.loads(manifest.read_text(encoding='utf-8'))['shards']
    hashes = {entry['filename']: entry['sha256'].lower() for entry in entries}
    if len(hashes) != len(entries):
        raise ValueError(f'Duplicate shard names in {manifest}')
    return hashes

def round0_pilot_shards():
    shards = collect_shards('round0_pilot')
    if len(shards) != 68:
        raise ValueError(f'Expected 68 Round 0 Pilot shards in {DATA_DIR}, found {len(shards)}')
    return shards

def run_s0_report(command, report_path, *args):
    previous_mtime = report_path.stat().st_mtime_ns if report_path.is_file() else None
    try:
        run_module('labyrinth_nn.data.s0_preburn', command, *args, '--out', report_path)
    except subprocess.CalledProcessError:
        if report_path.is_file() and report_path.stat().st_mtime_ns != previous_mtime:
            print(report_path.read_text(encoding='utf-8'))
        raise
    print(report_path.read_text(encoding='utf-8'))

def stop_simulator():
    subprocess.run(['bash', 'scripts/colab_setup_simulator.sh', '--port', str(PORT), '--stop-only'],
                   cwd=REPO_DIR, check=True)


def stop_after_error(error):
    try:
        stop_simulator()
    except (OSError, subprocess.CalledProcessError) as stop_error:
        error.add_note(f'Simulator teardown failed: {stop_error}')

import torch

with (REPO_DIR / 'pyproject.toml').open('rb') as project:
    version = tomllib.load(project)['project']['version']
print(f'Python {sys.version.split()[0]} | CPU count={os.cpu_count()} | CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print(f'labyrinth-nn {version} | source={REPO_SOURCE} | commit={ACTUAL_COMMIT or "unverified"}')
if LOCKED_COMMIT and not ACTUAL_COMMIT:
    print(f'Confirm the uploaded source matches declared commit {LOCKED_COMMIT}.')

In [ ]:
# Cell 2 · CPU · S0 zero-skip unit tests and damage proxy/data audit
if not CN_DB_PATH.is_file():
    raise FileNotFoundError(f'Upload the locked CN database to {CN_DB_PATH}')
tests = run_module('unittest', 'discover', '-s', 'tests', '-v',
                   check=False, stderr=subprocess.PIPE, text=True)
print(tests.stderr, end='')
tests.check_returncode()
if re.search(r'\bskipped=\d*[1-9]\d*', tests.stderr):
    raise AssertionError('S0 requires all unit tests to run without skips')

hash_args = []
if CN_DB_SHA256:
    hash_args.extend(['--db-sha256', CN_DB_SHA256])
if CATALOG_SHA256:
    hash_args.extend(['--catalog-sha256', CATALOG_SHA256])
run_s0_report('verify-data', STATUS_DIR / 's0_verify_data.json',
              '--db', CN_DB_PATH, '--catalog', CATALOG_PATH, *hash_args,
              '--samples', *round0_pilot_shards(), '--expected-shards', 68,
              '--expected-unique', 25000, '--min-nonwin-audit', 200)

In [ ]:
# Cell 3 · CPU · Stage offline cache on local SSD, start the simulator, and probe 64
persistent_dir = CACHE_DIR / 'persistent'
persistent_tar = CACHE_DIR / f'persistent_offline_{CN_DB_VERSION}.tar'
if not persistent_tar.is_file() and persistent_dir.is_dir():
    subprocess.run(['bash', 'scripts/pack_persistent_cache.sh', '--persistent-dir', persistent_dir,
                    '--out', persistent_tar], cwd=REPO_DIR, check=True)

setup = ['bash', 'scripts/colab_setup_simulator.sh', '--cache-dir', CACHE_DIR,
         '--db-path', CN_DB_PATH, '--db-version', CN_DB_VERSION, '--port', str(PORT)]
if persistent_tar.is_file():
    setup.extend(['--persistent-tar', persistent_tar])
if CN_DB_SHA256:
    setup.extend(['--db-sha256', CN_DB_SHA256])

try:
    subprocess.run(setup, cwd=REPO_DIR, check=True)
    if not CATALOG_PATH.is_file():
        run_module('labyrinth_nn.catalog.build_catalog', '--db', CN_DB_PATH, '--out', CATALOG_PATH)
    probe = run_module('labyrinth_nn.data.probe_simulator', '--catalog', CATALOG_PATH,
                       '--profile', PROFILE, '--url', SIM_URL, '--bench-count', 64,
                       check=False, stdout=subprocess.PIPE, text=True)
    PROBE_PATH.write_text(probe.stdout, encoding='utf-8')
    print(probe.stdout, end='')
    probe.check_returncode()
    report = json.loads(probe.stdout)
    if report['passed'] is not True or len(report['checks']) != 16:
        raise AssertionError('Expected all 16 simulator reconciliation checks to pass')
except (Exception, KeyboardInterrupt) as error:
    stop_after_error(error)
    raise

In [ ]:
# Cell 4 · CPU · S0 representative steady-state cost after warmup
OBSERVED_CPU_CC_PER_HOUR = 0.08  # Verify against the selected runtime.
try:
    run_s0_report('cpu-cost', STATUS_DIR / 's0_cpu_cost.json',
                  '--catalog', CATALOG_PATH, '--profile', S0_PROFILE, '--sampling', S0_SAMPLING,
                  '--url', SIM_URL, '--probe-bench-count', 64, '--probe-out', PROBE_PATH,
                  '--warmup-rows', 8, '--rows', 200, '--batch-size', 8, '--concurrency', 1,
                  '--timeout-sec', 120, '--sim-log', f'/tmp/pcr_calc_server_{PORT}.log',
                  '--cpu-cc-per-hour', OBSERVED_CPU_CC_PER_HOUR)
except (Exception, KeyboardInterrupt) as error:
    stop_after_error(error)
    raise

In [ ]:
# Cell 5 · CPU · S1/S3 segmented sampling and per-shard Drive sync
local_dir = Path('/tmp/limingjienn-samples')
local_dir.mkdir(parents=True, exist_ok=True)
manifest_name = f'{SAMPLE_PREFIX}.manifest.json'
failed_name = f'{SAMPLE_PREFIX}.failed.jsonl'

def sync_samples(source, destination):
    files = collect_shards(SAMPLE_PREFIX, source)
    immutable_names = {file.name for file in files}
    for file in files:
        target = destination / file.name
        if target.is_file() and file_sha256(file) != file_sha256(target):
            raise ValueError(f'Conflicting immutable shard: {target}')
    # Publish the manifest only after the shards and append-only failure log.
    files.extend(source / name for name in (failed_name, manifest_name) if (source / name).is_file())
    for file in files:
        target = destination / file.name
        if target.is_file():
            if file.name in immutable_names or file_sha256(file) == file_sha256(target):
                continue
            if file.name == manifest_name:
                incoming, existing = shard_hashes(file).items(), shard_hashes(target).items()
                if incoming < existing:
                    continue
                if not existing <= incoming:
                    raise ValueError(f'Conflicting manifest histories: {file} and {target}')
            else:
                incoming, existing = file.read_bytes(), target.read_bytes()
                if existing.startswith(incoming):
                    continue
                if not incoming.startswith(existing):
                    raise ValueError(f'Conflicting failed-sample logs: {file} and {target}')
        with tempfile.NamedTemporaryFile(dir=destination, prefix=f'.{file.name}.', delete=False) as temp:
            staged = Path(temp.name)
        try:
            shutil.copy2(file, staged)
            os.replace(staged, target)
        finally:
            staged.unlink(missing_ok=True)

sampling_args = ('--catalog', CATALOG_PATH, '--profile', PROFILE, '--sampling', SAMPLING,
                 '--seed', SEED, '--round-id', ROUND_ID, '--chunk-size', CHUNK_SIZE)
restored = False
try:
    sync_samples(DATA_DIR, local_dir)
    restored = True
    run_module('labyrinth_nn.data.sampler', *sampling_args, '--dry-run', 2000)
    sampling_run = run_module(
        'labyrinth_nn.data.sampler', *sampling_args, '--url', SIM_URL,
        '--out', local_dir / f'{SAMPLE_PREFIX}.jsonl', '--target-battles', TARGET_BATTLES,
        '--sync-dir', DATA_DIR, '--max-seconds', 10800, '--batch-size', 16, '--concurrency', 1,
        '--timeout-sec', 120, '--isolate-errors', '--resume', '--progress',
        stdout=subprocess.PIPE, text=True,
    )
    print(sampling_run.stdout, end='')
    if json.loads(sampling_run.stdout)['stopped_by_time_limit']:
        stop_simulator()
except (Exception, KeyboardInterrupt) as error:
    stop_after_error(error)
    raise
finally:
    if restored:
        pending_error = sys.exception()
        try:
            sync_samples(local_dir, DATA_DIR)
        except Exception as sync_error:
            if pending_error is None:
                stop_after_error(sync_error)
                raise
            pending_error.add_note(f'Final shard sync failed: {sync_error}')

In [ ]:
# Cell 6 · T4 · S0 timing or S2/S4 checkpoint-aware training
T4_MODE = 's0_timing'  # 's0_timing' or 'train'
OBSERVED_T4_CC_PER_HOUR = 1.17  # Confirm the selected T4 rate.
TRAIN_SEEDS = [42, 43]
MODEL_ARGS = ['--configs', 'm0', 'm1', 'm2a', 'm2b'] if ROUND == 0 else [
    '--config-files', 'configs/critic_m1.json', 'configs/critic_m2b.json',
    'configs/critic_m2b_r1.json', 'configs/critic_m2b_r1_dmg025.json',
]  # For S4, select the frozen winner and comparator configs here.

if T4_MODE not in {'s0_timing', 'train'}:
    raise ValueError('T4_MODE must be s0_timing or train')
if not torch.cuda.is_available():
    raise RuntimeError('Select a T4 GPU Runtime and rerun Cell 1')

if T4_MODE == 's0_timing':
    run_s0_report('t4-timing', STATUS_DIR / 's0_t4_timing.json',
                  '--catalog', CATALOG_PATH, '--config', 'configs/critic_m2b_r1.json',
                  '--samples', *round0_pilot_shards(), '--epochs', 2, '--device', 'cuda',
                  '--amp', '--pin-memory', '--group-by', 'context',
                  '--t4-cc-per-hour', OBSERVED_T4_CC_PER_HOUR)
else:
    if not TRAIN_PREFIXES or len(set(TRAIN_PREFIXES)) != len(TRAIN_PREFIXES):
        raise ValueError('Set TRAIN_PREFIXES to the distinct training shard prefixes')

    def verified_training_shards(prefix):
        files = collect_shards(prefix)
        manifest = DATA_DIR / f'{prefix}.manifest.json'
        if not files or not manifest.is_file():
            raise FileNotFoundError(f'Training requires shards and {manifest}')
        expected = shard_hashes(manifest)
        if set(expected) != {file.name for file in files}:
            raise ValueError(f'Shard filenames differ from {manifest}')
        for file in files:
            if file_sha256(file) != expected[file.name]:
                raise ValueError(f'Shard SHA256 differs from {manifest}: {file}')
        return files

    chunks = sorted({shard for prefix in TRAIN_PREFIXES for shard in verified_training_shards(prefix)})
    run_module('labyrinth_nn.train.train_critic', *MODEL_ARGS, '--seeds', *TRAIN_SEEDS,
               '--split-seed', 42, '--group-by', 'context', '--auto-resume', '--amp',
               '--device', 'cuda', '--pin-memory', '--catalog', CATALOG_PATH,
               '--samples', *chunks, '--out-dir', ABLATION_DIR)
    print((ABLATION_DIR / 'ablation_summary.md').read_text(encoding='utf-8'))

In [ ]:
# Cell 7 · T4 · Export ONNX and enforce bundle size and parity
WINNING_MODE = 'm2b'  # Set after reviewing the selection report.
WINNING_SEED = 42
WINNING_CKPT = ABLATION_DIR / WINNING_MODE / f'seed_{WINNING_SEED}' / 'best.pt'
BUNDLE_DIR = EXPORT_DIR / WINNING_MODE
run_module('labyrinth_nn.export.export_onnx', '--checkpoint', WINNING_CKPT,
           '--catalog', CATALOG_PATH, '--out-dir', BUNDLE_DIR)

manifest = json.loads((BUNDLE_DIR / 'manifest.json').read_text(encoding='utf-8'))
total_bytes = sum((BUNDLE_DIR / name).stat().st_size for name in ('tower_a.onnx', 'tower_b.onnx'))
max_abs_diff = manifest['validation']['max_abs_diff']
if total_bytes >= 10_000_000:
    raise AssertionError(f'ONNX towers have {total_bytes} bytes, limit is strictly below 10,000,000')
if not 0 <= max_abs_diff < 1e-4:
    raise AssertionError(f'ONNX parity failed: max_abs_diff={max_abs_diff}')
print(f'Exported {BUNDLE_DIR} | tower bytes={total_bytes} | max_abs_diff={max_abs_diff}')

In [ ]:
# Cell 8 · CPU · Path B replay and sealed evaluation with a frozen checkpoint
from labyrinth_nn.catalog.schema import FeatureCatalog
from labyrinth_nn.env.run_env import PathBReplayPolicy, compare_policies, load_replays

REPLAY_PATH = DRIVE_ROOT / 'replays.jsonl'  # Optional recorded enter/map_list JSONL
if REPLAY_PATH.exists():
    catalog = FeatureCatalog.load(CATALOG_PATH)
    episodes = load_replays(REPLAY_PATH)
    reports = compare_policies(
        episodes,
        path_b=PathBReplayPolicy(target_score=53000),
        settings=catalog.score_settings,
        catalog=catalog,
        repeats=5,
    )
    for name, rep in reports.items():
        print(name, rep)
else:
    print(f'Place recorded opening maps at {REPLAY_PATH} to run Path B replay comparison.')

# Set these paths after the panel, model and selection rules have been frozen.
SEALED_SHARDS = []  # All sealed JSONL/Parquet paths, outside TRAIN_PREFIXES.
FROZEN_CHECKPOINT = None

if bool(SEALED_SHARDS) != bool(FROZEN_CHECKPOINT):
    raise ValueError('Set both SEALED_SHARDS and FROZEN_CHECKPOINT to score the frozen panel')

if SEALED_SHARDS and FROZEN_CHECKPOINT:
    from torch.utils.data import DataLoader
    from labyrinth_nn.catalog.schema import FeatureCatalog
    from labyrinth_nn.data.dataset import BattleDataset, collate_battle_batch, mask_gold_sample
    from labyrinth_nn.data.sample_schema import read_samples
    from labyrinth_nn.models.critic import CriticConfig, TwoTowerBattleCritic
    from labyrinth_nn.train.metrics import evaluate_predictions
    from labyrinth_nn.train.train_critic import predict_records

    catalog = FeatureCatalog.load(CATALOG_PATH)
    state = torch.load(FROZEN_CHECKPOINT, map_location='cpu', weights_only=True)
    if state['catalog_sha256'] != (catalog.catalog_sha256 or catalog.compute_sha256()):
        raise ValueError('Sealed panel catalog differs from the frozen checkpoint')
    config = CriticConfig.from_dict(state['config'], catalog)
    model = TwoTowerBattleCritic(config)
    model.load_state_dict(state['model'])
    missing = [Path(shard) for shard in SEALED_SHARDS if not Path(shard).is_file()]
    if missing:
        raise FileNotFoundError(f'Missing sealed panel shards: {missing}')
    train_paths = {shard.resolve() for prefix in TRAIN_PREFIXES for shard in collect_shards(prefix)}
    if train_paths.intersection(Path(shard).resolve() for shard in SEALED_SHARDS):
        raise ValueError('Sealed shards overlap the training inputs')
    samples = [sample for shard in sorted(map(Path, SEALED_SHARDS)) for sample in read_samples(shard)]
    if (not samples or len({s.sample_id for s in samples}) != len(samples)
            or any(s.db_version != catalog.db_version or not sum(s.outcome_counts) for s in samples)):
        raise ValueError('Sealed panel must be nonempty, unique, labeled and on the locked DB version')

    def score(rows):
        dataset = BattleDataset(catalog, rows, roster_max_len=config.roster_max_len,
                                relic_max_len=config.relic_max_len)
        loader = DataLoader(dataset, batch_size=config.batch_size, collate_fn=collate_battle_batch)
        return evaluate_predictions(predict_records(model, loader, 'cpu'))

    report = {'metrics': score(samples),
              'counterfactual_gold_unknown': score([mask_gold_sample(sample) for sample in samples])}
    sealed_report = RUNS_DIR / 'sealed_test_metrics.json'
    sealed_report.write_text(json.dumps(report, indent=2), encoding='utf-8')
    print('Sealed evaluation:', sealed_report)
else:
    print('Set SEALED_SHARDS and FROZEN_CHECKPOINT to evaluate the frozen panel.')

In [ ]:
# Cell 9 · CPU / T4 · Stop the simulator, update the receipt, and release the runtime
UPDATE_RECEIPT = False
AUTO_UNASSIGN = True

CURRENT_STAGE = 'S0'
STAGE_NAME = None
STAGE_STATUS = None
STAGE_CC = None
OBSERVED_BALANCE_CC = None
ARTIFACT_KEY = None
ARTIFACT_VALUE = None
BLOCKER = None
CLEAR_BLOCKER = False
NOTE = None

try:
    stop_simulator()
    if UPDATE_RECEIPT:
        commit = LOCKED_COMMIT or ACTUAL_COMMIT
        if not commit:
            raise ValueError('Set LOCKED_COMMIT to the verified code revision before updating the receipt')
        if (ARTIFACT_KEY is None) != (ARTIFACT_VALUE is None):
            raise ValueError('Set ARTIFACT_KEY and ARTIFACT_VALUE together')
        receipt = STATUS_DIR / 'latest_receipt.json'
        args = ['--receipt', receipt, '--locked-commit', commit]
        for flag, value in (
            ('--current-stage', CURRENT_STAGE), ('--stage-name', STAGE_NAME),
            ('--stage-status', STAGE_STATUS), ('--stage-cc', STAGE_CC),
            ('--observed-balance-cc', OBSERVED_BALANCE_CC), ('--artifact-key', ARTIFACT_KEY),
            ('--artifact-value', ARTIFACT_VALUE), ('--blocker', BLOCKER), ('--note', NOTE),
        ):
            if value is not None:
                args.extend([flag, value])
        if CLEAR_BLOCKER:
            args.append('--clear-blocker')
        run_module('labyrinth_nn.data.s0_preburn', 'update-receipt', *args)
        print(receipt.read_text(encoding='utf-8'))
finally:
    if AUTO_UNASSIGN:
        from google.colab import runtime

        runtime.unassign()